In [1]:
import pygame
import numpy as np
import sys

pygame.init()

WIDTH, HEIGHT = 900, 700
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Real Neural Activity — c37")
clock = pygame.time.Clock()

spike_times = np.load("data/valid_spike_times.npy")
cluster_labels = np.load("data/cluster_labels.npy")
fs = 30000

# convert spike times to milliseconds, playback speed multiplier
playback_speed = 5  # slow down real-time by this factor
spike_times_ms = (spike_times / fs) * 1000 * playback_speed

# assign each cluster a distinct color
unique_clusters = np.unique(cluster_labels)
colors = [
    (255, 90, 90), (90, 200, 255), (255, 220, 90), (150, 255, 150),
    (255, 150, 255), (150, 150, 255), (255, 180, 90), (180, 255, 220),
    (220, 150, 255), (255, 255, 150)
]
cluster_colors = {c: colors[i % len(colors)] for i, c in enumerate(unique_clusters)}

# each cluster gets a fixed screen position (a "neuron node")
np.random.seed(0)
cluster_positions = {
    c: (np.random.randint(100, WIDTH-100), np.random.randint(100, HEIGHT-100))
    for c in unique_clusters
}

particles = []  # each: [x, y, radius, color, alpha]

start_ticks = pygame.time.get_ticks()
spike_idx = 0
sorted_order = np.argsort(spike_times_ms)
spike_times_ms = spike_times_ms[sorted_order]
cluster_labels_sorted = cluster_labels[sorted_order]

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    now_ms = pygame.time.get_ticks() - start_ticks

    # trigger any spikes whose time has come
    while spike_idx < len(spike_times_ms) and spike_times_ms[spike_idx] <= now_ms:
        c = cluster_labels_sorted[spike_idx]
        x, y = cluster_positions[c]
        particles.append([x, y, 5, cluster_colors[c], 255])
        spike_idx += 1

    if spike_idx >= len(spike_times_ms):
        spike_idx = 0
        start_ticks = pygame.time.get_ticks()  # loop the recording

    screen.fill((10, 10, 20))

    # draw and update particles (expanding, fading circles)
    new_particles = []
    for p in particles:
        x, y, r, color, alpha = p
        surf = pygame.Surface((r*2, r*2), pygame.SRCALPHA)
        pygame.draw.circle(surf, (*color, alpha), (r, r), r)
        screen.blit(surf, (x - r, y - r))

        r += 1.5
        alpha -= 8
        if alpha > 0:
            new_particles.append([x, y, r, color, alpha])
    particles = new_particles

    pygame.display.flip()
    clock.tick(60)

pygame.quit()
sys.exit()

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


SystemExit: 

/Users/rishirajmanna/Desktop/projects/spike-visualizer/venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from vpython import sphere, vector, color, scene, rate
import numpy as np
import time

scene.title = "Real Neural Activity in 3D — c37"
scene.width = 1000
scene.height = 700
scene.background = vector(0.02, 0.02, 0.05)
scene.forward = vector(-1, -0.3, -1)  # initial camera angle

spike_times = np.load("data/valid_spike_times.npy")
cluster_labels = np.load("data/cluster_labels.npy")
fs = 30000

playback_speed = 5
spike_times_sec = (spike_times / fs) * playback_speed

sorted_order = np.argsort(spike_times_sec)
spike_times_sec = spike_times_sec[sorted_order]
cluster_labels_sorted = cluster_labels[sorted_order]

unique_clusters = np.unique(cluster_labels)
colors = [
    color.red, color.cyan, color.yellow, color.green,
    color.magenta, color.blue, color.orange, color.white,
    vector(0.7, 0.3, 1), vector(1, 0.7, 0.3)
]
cluster_colors = {c: colors[i % len(colors)] for i, c in enumerate(unique_clusters)}

np.random.seed(0)
cluster_positions = {
    c: vector(
        np.random.uniform(-8, 8),
        np.random.uniform(-5, 5),
        np.random.uniform(-8, 8),
    )
    for c in unique_clusters
}

# draw a small permanent marker for each neuron's position
for c in unique_clusters:
    sphere(pos=cluster_positions[c], radius=0.15, color=cluster_colors[c], opacity=0.4)

pulses = []  # each: {sphere_obj, growth_rate}

spike_idx = 0
start_time = time.time()

while True:
    rate(60)

    scene.forward = vector(
        scene.forward.x * np.cos(0.002) - scene.forward.z * np.sin(0.002),
        scene.forward.y,
        scene.forward.x * np.sin(0.002) + scene.forward.z * np.cos(0.002),
    )  # slow auto-rotate camera

    now = time.time() - start_time

    while spike_idx < len(spike_times_sec) and spike_times_sec[spike_idx] <= now:
        c = cluster_labels_sorted[spike_idx]
        pos = cluster_positions[c]
        pulse = sphere(pos=pos, radius=0.2, color=cluster_colors[c], opacity=1.0, emissive=True)
        pulses.append(pulse)
        spike_idx += 1

    if spike_idx >= len(spike_times_sec):
        spike_idx = 0
        start_time = time.time()

    new_pulses = []
    for p in pulses:
        p.radius += 0.03
        p.opacity -= 0.015
        if p.opacity > 0:
            new_pulses.append(p)
        else:
            p.visible = False
    pulses = new_pulses

In [6]:
pip install "setuptools<81"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.6 MB/s eta 0:00:008.7 MB/s eta 0:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 83.0.0
    Uninstalling setuptools-83.0.0:
      Successfully uninstalled setuptools-83.0.0

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
